# Customer Purchase Propensity – CRISP-DM
Dự báo xác suất hộ gia đình có ít nhất một basket trong 7 ngày kế tiếp.

Notebook đọc artifact từ pipeline sản xuất để trình bày kết quả tái lập.

## Phase 1 – Business Understanding
Đơn vị dự báo: `(household_key, snapshot_day)`. Target: ít nhất một basket trong `(snapshot_day, snapshot_day + 7]`. Dùng score để ranking, không diễn giải như hiệu quả causal của coupon.

In [ ]:
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
ARTIFACTS = ROOT / 'artifacts' / 'propensity'
metrics = json.loads((ARTIFACTS / 'metrics.json').read_text(encoding='utf-8'))
metrics['data_profile']

## Phase 2 – Data Understanding
Kiểm tra grain, khoảng thời gian, null, duplicate và coverage demographic.

In [ ]:
pd.DataFrame([metrics['data_quality']]).T.rename(columns={0: 'value'})

## Phase 3 – Data Preparation
Feature kết thúc tại snapshot, label bắt đầu sau snapshot. Split theo thời gian và có embargo một horizon giữa các tập.

In [ ]:
metrics['split_manifest']

## Phase 4 – Modeling
So sánh Logistic Regression và Histogram Gradient Boosting trên validation; chọn theo PR-AUC rồi calibration bằng Platt scaling.

In [ ]:
pd.DataFrame(metrics['validation_metrics']).T

## Phase 5 – Evaluation
Test chỉ mở sau khi khóa model và threshold.

In [ ]:
pd.Series(metrics['test_metrics'], name='test').to_frame()

In [ ]:
cal = pd.read_csv(ARTIFACTS / 'test_calibration.csv')
ax = cal.plot(x='mean_predicted_probability', y='observed_purchase_rate', marker='o', xlim=(0, 1), ylim=(0, 1), grid=True, legend=False, title='Calibration on temporal holdout')
ax.plot([0, 1], [0, 1], '--', color='gray')

In [ ]:
importance = pd.read_csv(ARTIFACTS / 'feature_importance.csv').head(15)
importance.sort_values('importance_mean').plot.barh(x='feature', y='importance_mean', legend=False, title='Permutation importance (PR-AUC)')

## Phase 6 – Deployment
Chạy `python src/propensity_pipeline.py score` để tạo score tại raw day mới nhất. Theo dõi schema/freshness, score drift, PR-AUC, Brier, calibration và lift@K.

In [ ]:
scores = pd.read_csv(ARTIFACTS / 'latest_scores.csv')
scores.head(10)